# H&N vs Lung – Rybrevant FASPRO DDD segmentation (L2/L3, weekly)

**Goal:** split **Rybrevant FASPRO DDD sales**, per account (L2/L3) and per week, into **H&N only / H&N + Lung / Lung only**.
Only DDD is split. All other sources are used to decide which segment an account is in.

| Universe | Sources |
|---|---|
| **H&N** | LE.AI H&N HCP output (`Rybrevant_HNSCC_HV_CLAIMS`). Careset: pending, not used yet |
| **Lung (FASPRO)** | DDD, Xponent, MX claims, RX claims, McKesson EMR, Chargemaster |

**H&N presence** = LE.AI H&N HCPs, **or** Chargemaster HNSCC encounters, **or** an H&N diagnosis in McKesson EMR.

**Weekly rule (Ravi):** for each account and each week with a FASPRO DDD sale,
- **Lung present** = the account is in the historical FASPRO Lung universe from Xponent / MX / RX / McKesson / Chargemaster (YTD), **or** it bought FASPRO in **DDD in the 13 weeks before** the sale. The 13-week look-back uses **DDD only**.
- **H&N present** = the account is in the H&N universe.
- That week's FASPRO DDD units go to: **H&N only / H&N + Lung / Lung only**.
- Accounts with neither are reported as **Unclassified**.

**Scope:**
- **Rybrevant FASPRO only** (no Rybrevant IV, no Lazcluze), in every source.
- Window: **YTD 2026 = 2026-01-01 → 2026-10-22**. FASPRO launched end of Dec 2025.

**Levels:** L0 HCP · L1 outlet (DDD) · L2 hospital/account · L3 larger account. Everything is rolled up to **L2/L3** with OPAL (L2 when available, else L3).

Sections: 1 setup · 2 window · 3 lookups (NPI → J&J ID → account) · 4 H&N universe · 5 FASPRO Lung activity by source · 6 historical account segments (YTD) · 7 weekly DDD segmentation · 8 account segment insights + figures · 9 save

## 1. Setup

### What this cell does: settings, helpers, and loading
- **All settings are here:**
  - `START_DATE` / `END_DATE`: **YTD 2026**, 2026-01-01 → 2026-10-22 ("today").
  - `LOOKBACK_WEEKS = 13`: the weekly rule looks at **DDD only**, in the 13 weeks **before** the sale.
  - `FASPRO_NDC`: the FASPRO NDCs from the data-sources file.
  - `MCKESSON_FASPRO_ONLY = True`: keep only McKesson alerts whose name mentions FASPRO (team decision).
  - `CM_TIME_FRAME`: which Chargemaster period to use (`CY-2026` = FASPRO year to date).
  - `CM_HNSCC_AS_HN = True`: Chargemaster HNSCC encounters count as H&N presence (team decision).
  - `MCKESSON_HN_DX_AS_HN = True`: an H&N diagnosis (HNSCC / head & neck) in McKesson counts as H&N presence.
  - `HN_ACCOUNT_SOURCE`: how LE.AI HCPs are placed on accounts.
    - `"opal"`: OPAL `L2_L3`.
    - `"cohort"`: LE.AI's own mapped account when L2/L3.
  - `RECENT_WEEKS`: how many latest weeks to show in detail.
- **Helpers:**
  - `clean()`: turns empty text and placeholders like `"-"` into null.
  - `pick()`: finds a column even with hidden line breaks or odd spacing.
  - `week_of()`: the Monday of a date's week; all sources use the same week.

In [ ]:
import datetime as dt
import re
import dataiku
import dataiku.spark as dkuspark
from pyspark.sql import SparkSession, SQLContext, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
sqlContext = SQLContext(spark.sparkContext)

# ---------------- settings ----------------
DATASETS = {
    "hn_cohort":    "Rybrevant_HNSCC_HV_CLAIMS",
    "mx":           "MX_RYB_CLAIMS_copy",
    "rx":           "RYB_LINK_CLAIMS_copy",
    "dmn":          "dmn_cust_copy",
    "opal":         "OPAL_Affiliation_Levels",
    "ddd":          "RYB_DDD_Sales_copy",
    "xpon":         "RYB_Xpon_Sales_copy",
    "mckesson":     "Mckesson_GPO_DATA_copy",
    "chargemaster": "ChargeMaster_WeeklyReporting_20261012_v2_copy",
}
START_DATE = "2026-01-01"               # YTD 2026 (FASPRO launched end of Dec 2025)
END_DATE = "2026-10-22"                 # "today"
LOOKBACK_WEEKS = 13                     # Ravi: DDD FASPRO purchases in the 13 weeks BEFORE the sale -> Lung
FASPRO_NDC = ["57894051001", "57894051401", "57894051501", "57894052201"]   # RYB_HN_Data_Sources.csv
MCKESSON_FASPRO_ONLY = True
CM_TIME_FRAME = "CY-2026"
CM_HNSCC_AS_HN = True                   # team decision: Chargemaster HNSCC encounters = H&N presence
MCKESSON_HN_DX_AS_HN = True             # an H&N diagnosis in McKesson EMR = H&N presence (Ravi summary)
HN_ACCOUNT_SOURCE = "opal"
RECENT_WEEKS = 3
# -------------------------------------------

MISSING = ["", "-", "--", "NA", "N/A", "NULL", "NONE", "NAN"]

def load(name):
    return dkuspark.get_dataframe(sqlContext, dataiku.Dataset(name))

def clean(col):
    """trim; empty text and placeholders like "-" -> null"""
    s = F.trim(col.cast("string"))
    return F.when(~F.upper(s).isin(MISSING), s)

def pick(d, name):
    """find a column even if its real name has extra spaces, line breaks or different punctuation/case"""
    key = lambda x: re.sub(r"[^0-9a-z]", "", x.lower())
    found = {key(x): x for x in d.columns}
    if key(name) not in found:
        raise KeyError(f"column '{name}' not found. Available: {d.columns}")
    return F.col(f"`{found[key(name)]}`")

def week_of(col):
    """Monday of the week (same week definition for every source)"""
    return F.to_date(F.date_trunc("week", F.to_date(col)))

def is_faspro_ndc(col):
    return F.regexp_replace(F.trim(col.cast("string")), "[^0-9]", "").isin(FASPRO_NDC)

df = {k: load(v) for k, v in DATASETS.items()}
for k, v in df.items():
    print(f"{k:13s} {DATASETS[k]:48s} rows={v.count():>12,}")

## 2. Date window

### What this cell does: define the window and check each source's dates
- Window = **YTD 2026**: `START_DATE` 2026-01-01 → `END_DATE` 2026-10-22.
- Shows each source's earliest and latest date.
- **Look at the latest dates:** sources that end earlier than others have fewer recent weeks.
  - That matters for the 13-week rule in the latest weeks.
- The 13-week DDD look-back for early-January sales reaches into Oct–Dec 2025. It uses whatever DDD history exists, but FASPRO only launched end of Dec 2025.

In [ ]:
n_days = (dt.datetime.strptime(END_DATE, "%Y-%m-%d") - dt.datetime.strptime(START_DATE, "%Y-%m-%d")).days + 1
print(f"Window: {START_DATE} .. {END_DATE} (YTD, {n_days} days = {n_days / 7:.1f} weeks)\n")

for key, colname in [("ddd", "caln_dt"), ("xpon", "caln_dt"), ("mx", "service_date"), ("rx", "fill_date"),
                     ("mckesson", "diagnosis_date")]:
    r = df[key].agg(F.min(F.to_date(F.col(colname))).alias("min"), F.max(F.to_date(F.col(colname))).alias("max")).collect()[0]
    print(f"{key:9s} {colname:15s} {r['min']} .. {r['max']}")

cm = df["chargemaster"]
print("\nChargemaster time frames available:")
cm.groupBy(pick(cm, "Cadence").alias("cadence"), pick(cm, "Time Frame Descriptive").alias("time_frame")).count() \
  .orderBy("cadence", "time_frame").show(50, False)

def in_window(col):
    return F.to_date(col).between(F.lit(START_DATE), F.lit(END_DATE))

## 3. Lookups: NPI → J&J ID → L2/L3 account

### What this cell does: build the ID lookups used by every source
- **NPI → J&J ID** (dmn_cust): if an NPI has several `jnj_id`s, keep the latest `efftv_dt` (data-sources rule).
- **Any J&J ID → L2/L3 account** (OPAL), in this order:
  1. it is an **HCP** (OPAL column `0`) → its `L2_L3`;
  2. it is an **outlet** (column `1`) → the most common `L2_L3` on its rows;
  3. it is itself an **L2** (column `2`) or **L3** (column `3`) account → itself.
- **Account level:** L2 if the ID appears in column `2`, otherwise L3.
- `to_account(table, "id column")` adds `account_id` to any table. Used for DDD, Xponent, claims, McKesson and Chargemaster.

In [ ]:
npi_jnj = (df["dmn"].select(clean(F.col("npi_id")).alias("npi"), clean(F.col("jnj_id")).alias("jnj_id"),
                            F.to_date("efftv_dt").alias("efftv_dt"))
           .filter("npi IS NOT NULL AND jnj_id IS NOT NULL")
           .withColumn("rk", F.row_number().over(Window.partitionBy("npi").orderBy(F.desc_nulls_last("efftv_dt"), "jnj_id")))
           .filter("rk = 1").drop("rk", "efftv_dt").cache())

o = df["opal"].select(*[clean(F.col(f"`{x}`")).alias(n) for x, n in
                        [("0", "hcp"), ("1", "l1"), ("2", "l2"), ("3", "l3"), ("L2_L3", "account_id")]])
by_hcp = (o.filter("hcp IS NOT NULL AND account_id IS NOT NULL")
           .select(F.col("hcp").alias("jnj_id"), "account_id", F.lit(1).alias("prio")))
by_l1 = (o.filter("l1 IS NOT NULL AND account_id IS NOT NULL").groupBy(F.col("l1").alias("jnj_id"), "account_id").count()
          .withColumn("rk", F.row_number().over(Window.partitionBy("jnj_id").orderBy(F.desc("count"), "account_id")))
          .filter("rk = 1").select("jnj_id", "account_id", F.lit(2).alias("prio")))
by_l2 = o.filter("l2 IS NOT NULL").select(F.col("l2").alias("jnj_id"), F.col("l2").alias("account_id"), F.lit(3).alias("prio"))
by_l3 = o.filter("l3 IS NOT NULL").select(F.col("l3").alias("jnj_id"), F.col("l3").alias("account_id"), F.lit(4).alias("prio"))
id_to_account = (by_hcp.unionByName(by_l1).unionByName(by_l2).unionByName(by_l3).distinct()
                 .withColumn("rk", F.row_number().over(Window.partitionBy("jnj_id").orderBy("prio", "account_id")))
                 .filter("rk = 1").select("jnj_id", "account_id").cache())
account_level = (by_l2.select("account_id", F.lit("L2").alias("level"))
                 .unionByName(by_l3.select("account_id", F.lit("L3").alias("level")))
                 .groupBy("account_id").agg(F.min("level").alias("account_level")).cache())

def to_account(d, id_col):
    return d.join(id_to_account.withColumnRenamed("jnj_id", id_col), id_col, "left")

print("NPIs in lookup:", f"{npi_jnj.count():,}", "| J&J IDs resolvable to an account:", f"{id_to_account.count():,}")

## 4. H&N universe (LE.AI)

### What this cell does: put the LE.AI H&N HCPs on their L2/L3 accounts
- Reads the LE.AI file. `"-"` in `J&J ID` is treated as missing; the J&J ID then comes from dmn_cust via the NPI.
- Account per HCP depends on `HN_ACCOUNT_SOURCE`:
  - `"opal"`: OPAL `L2_L3`, the same rule as the Lung side;
  - `"cohort"`: LE.AI's own `Mapped Account JNJ ID` when its level is L2/L3, else OPAL.
- Result `hn_accounts`: one row per account with:
  - the number of H&N HCPs;
  - their **H&N treating patients (2 years)** and **Origami-4 eligible patients (2 years)**, from LE.AI, summed per account and used for the insights.
- Prints how many HCPs reach an account, under both options.
- LE.AI has no dates (it is a 2-year cohort), so H&N presence is the **same for every week**.

In [ ]:
h = df["hn_cohort"]
coh = (h.select(clean(pick(h, "NPI")).alias("npi"), clean(pick(h, "J&J ID")).alias("cohort_jnj_id"),
                clean(pick(h, "Mapped Account JNJ ID")).alias("cohort_account_id"),
                F.upper(clean(pick(h, "Mapped Account JNJ Level"))).alias("cohort_account_level"),
                clean(pick(h, "Mapped Account Name")).alias("cohort_account_name"),
                clean(pick(h, "Treating Patient Volume - HV (2 years)")).cast("double").alias("hn_pts_2y"),
                clean(pick(h, "Origami-4 Eligible Patients (2 years)")).cast("double").alias("origami4_pts_2y"))
         .join(npi_jnj.select("npi", F.col("jnj_id").alias("dmn_jnj_id")), "npi", "left")
         .withColumn("jnj_id", F.coalesce("cohort_jnj_id", "dmn_jnj_id")))
coh = to_account(coh, "jnj_id").withColumnRenamed("account_id", "opal_account_id")
coh = coh.withColumn("cohort_account", F.when(F.col("cohort_account_level").isin("L2", "L3"), F.col("cohort_account_id")))
for src, acc in [("opal", F.col("opal_account_id")), ("cohort", F.coalesce("cohort_account", "opal_account_id"))]:
    n = coh.select("npi").distinct().count(); m = coh.filter(acc.isNotNull()).select("npi").distinct().count()
    print(f"H&N account source = {src:6s}: LE.AI HCPs placed on an account {m:,} of {n:,}")

acc_col = F.col("opal_account_id") if HN_ACCOUNT_SOURCE == "opal" else F.coalesce("cohort_account", "opal_account_id")
hcp_acc = (coh.withColumn("account_id", acc_col).filter("account_id IS NOT NULL")
              .groupBy("account_id", "npi").agg(F.max("hn_pts_2y").alias("hn_pts_2y"),
                                                F.max("origami4_pts_2y").alias("origami4_pts_2y")))
hn_accounts = (hcp_acc.groupBy("account_id")
                      .agg(F.countDistinct("npi").alias("hn_hcps"), F.sum("hn_pts_2y").alias("hn_pts_2y"),
                           F.sum("origami4_pts_2y").alias("origami4_pts_2y")).cache())
print("H&N accounts:", f"{hn_accounts.count():,}")

## 5. FASPRO Lung activity by source

### What this cell does: collect every FASPRO activity and place it on an account and a week
- One long table, `activity`: account, week, source, amount. Sources:
  - **DDD**: `prod_nm` contains FASPRO; DDD ID → account; amount = equivalized units.
  - **Xponent**: `prod_nm` contains FASPRO; HCP J&J ID → account; amount = TRx.
  - **MX claims**: FASPRO NDCs; `rendering_npi` → J&J ID → account; amount = infusions (patient + day + NDC).
  - **RX claims**: FASPRO NDCs; only patients **not** in the MX FASPRO claims; `prescriber_npi` → account; amount = fills.
  - **McKesson**: alerts mentioning FASPRO (`MCKESSON_FASPRO_ONLY`); `physician_npi` → account; week of `diagnosis_date`; amount = alerts.
  - **Chargemaster**:
    - rows for `CM_TIME_FRAME`; `NPI` → J&J ID → account; amount = `# EncountersNSCLC`.
    - It is an **annual total with no week**, so it counts as Lung activity for every week of that year.
- Shows, per source: rows, amount, and the share that reached an account.
- **Note:** Chargemaster has no product column, so it can't be limited to FASPRO.

In [ ]:
parts = []

# DDD
ddd = (df["ddd"].filter(F.upper("prod_nm").contains("FASPRO"))
                .select(F.col("jnj_id").cast("string").alias("src_id"), F.to_date("caln_dt").alias("day"),
                        week_of("caln_dt").alias("week"), F.col("equiv_uns").cast("double").alias("amount")))
ddd = to_account(ddd.withColumnRenamed("src_id", "jnj_id"), "jnj_id").cache()
parts.append(ddd.select("account_id", "day", "week", F.lit("DDD").alias("source"), "amount", "jnj_id"))

# Xponent
xp = (df["xpon"].filter(F.upper("prod_nm").contains("FASPRO"))
                .select(clean(F.col("jnj_id")).alias("jnj_id"), F.to_date("caln_dt").alias("day"), week_of("caln_dt").alias("week"),
                        F.col("equ_trx").cast("double").alias("amount")))
parts.append(to_account(xp, "jnj_id").select("account_id", "day", "week", F.lit("XPONENT").alias("source"), "amount", "jnj_id"))

# MX claims
mx = (df["mx"].filter(is_faspro_ndc(F.col("ndc")))
              .select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"), F.to_date("service_date").alias("day"),
                      F.col("ndc"), clean(F.col("rendering_npi")).alias("npi")))
mx_patients = mx.select("patient_id").distinct()
mx_inf = (mx.dropDuplicates(["patient_id", "day", "ndc"]).join(npi_jnj, "npi", "left")
            .select("jnj_id", "day", week_of("day").alias("week"), F.lit(1.0).alias("amount")))
parts.append(to_account(mx_inf, "jnj_id").select("account_id", "day", "week", F.lit("MX").alias("source"), "amount", "jnj_id"))

# RX claims (patients not in MX FASPRO claims)
rx = (df["rx"].filter(is_faspro_ndc(F.col("ndc")))
              .select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"), "claim_id", F.to_date("fill_date").alias("day"),
                      clean(F.col("prescriber_npi")).alias("npi"))
              .join(mx_patients, "patient_id", "left_anti").dropDuplicates(["claim_id"])
              .join(npi_jnj, "npi", "left")
              .select("jnj_id", "day", week_of("day").alias("week"), F.lit(1.0).alias("amount")))
parts.append(to_account(rx, "jnj_id").select("account_id", "day", "week", F.lit("RX").alias("source"), "amount", "jnj_id"))

# McKesson
mk = df["mckesson"]
if MCKESSON_FASPRO_ONLY:
    mk = mk.filter(F.upper("alert_name").contains("FASPRO"))
mk = mk.filter(F.to_date("diagnosis_date").between(F.lit(START_DATE), F.lit(END_DATE)))
dx_txt = F.upper(F.coalesce(F.col("diagnosis"), F.lit("")))
mk_hn = (mk.filter(dx_txt.contains("HNSCC") | dx_txt.contains("HEAD") | dx_txt.contains("NECK"))
           .select(clean(F.col("physician_npi")).alias("npi")).join(npi_jnj, "npi", "left"))
mk_hn = to_account(mk_hn, "jnj_id").filter("account_id IS NOT NULL").select("account_id").distinct()
mk = (mk.select(clean(F.col("physician_npi")).alias("npi"), F.to_date("diagnosis_date").alias("day"),
                week_of("diagnosis_date").alias("week"), F.lit(1.0).alias("amount"))
        .join(npi_jnj, "npi", "left"))
parts.append(to_account(mk, "jnj_id").select("account_id", "day", "week", F.lit("MCKESSON").alias("source"), "amount", "jnj_id"))

# Chargemaster (annual, no week)
cm_rows = (cm.filter(F.trim(pick(cm, "Time Frame Descriptive")) == CM_TIME_FRAME)
             .select(clean(pick(cm, "NPI")).alias("npi"),
                     clean(pick(cm, "# EncountersNSCLC")).cast("double").alias("nsclc"),
                     clean(pick(cm, "# EncountersHNSCC")).cast("double").alias("hnscc"))
             .join(npi_jnj, "npi", "left"))
cm_acc = to_account(cm_rows, "jnj_id").cache()
parts.append(cm_acc.select("account_id", F.lit(None).cast("date").alias("day"), F.lit(None).cast("date").alias("week"), F.lit("CHARGEMASTER").alias("source"),
                           F.col("nsclc").alias("amount"), "jnj_id"))

activity = parts[0]
for p in parts[1:]:
    activity = activity.unionByName(p)
activity = activity.filter(F.coalesce("amount", F.lit(0.0)) > 0).cache()

(activity.groupBy("source")
         .agg(F.count("*").alias("rows"), F.round(F.sum("amount"), 1).alias("amount"),
              F.round(100 * F.sum(F.when(F.col("jnj_id").isNotNull(), 1).otherwise(0)) / F.count("*"), 1).alias("pct_with_jnj_id"),
              F.round(100 * F.sum(F.when(F.col("account_id").isNotNull(), F.col("amount")).otherwise(0)) / F.sum("amount"), 1)
               .alias("pct_amount_on_account"),
              F.min("week").alias("first_week"), F.max("week").alias("last_week"))
         .orderBy("source").show(truncate=False))
print("Chargemaster", CM_TIME_FRAME, "HCPs with HNSCC encounters:", cm_acc.filter("hnscc > 0").select("npi").distinct().count())

## 6. Account segments (whole 52 weeks)

### What this cell does: historical segmentation of every account, YTD 2026 (flow chart box 6)
- **Lung universe:** any FASPRO activity at the account YTD, in any source (DDD, Xponent, MX, RX, McKesson, Chargemaster).
- **H&N universe:**
  - LE.AI HCPs;
  - plus Chargemaster HNSCC encounters (`CM_HNSCC_AS_HN`);
  - plus H&N diagnoses in McKesson (`MCKESSON_HN_DX_AS_HN`).
- **Segment:** H&N only / H&N + Lung / Lung only.
- **Outputs:**
  1. segment counts by L2 / L3, with totals;
  2. % of accounts per segment;
  3. FASPRO activity per source per segment.

In [ ]:
in_win = F.col("day").isNull() | F.col("day").between(F.lit(START_DATE), F.lit(END_DATE))   # real date, not week start
lung_acct = (activity.filter(in_win & F.col("account_id").isNotNull())
             .groupBy("account_id").pivot("source", ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER"])
             .agg(F.round(F.sum("amount"), 2)).fillna(0)
             .withColumn("lung_present", F.lit(True)))
hn_set = hn_accounts.withColumn("hn_src", F.lit("LE.AI"))
if CM_HNSCC_AS_HN:
    hn_set = hn_set.unionByName(cm_acc.filter("hnscc > 0 AND account_id IS NOT NULL").select("account_id").distinct()
                                .withColumn("hn_hcps", F.lit(0)).withColumn("hn_pts_2y", F.lit(0.0)).withColumn("origami4_pts_2y", F.lit(0.0)).withColumn("hn_src", F.lit("CHARGEMASTER_HNSCC")))
if MCKESSON_HN_DX_AS_HN:
    hn_set = hn_set.unionByName(mk_hn.withColumn("hn_hcps", F.lit(0)).withColumn("hn_pts_2y", F.lit(0.0)).withColumn("origami4_pts_2y", F.lit(0.0)).withColumn("hn_src", F.lit("MCKESSON_HN_DX")))
print("H&N universe accounts by source:")
hn_set.groupBy("hn_src").agg(F.countDistinct("account_id").alias("accounts")).show(truncate=False)
hn_set = (hn_set.groupBy("account_id").agg(F.max("hn_hcps").alias("hn_hcps"), F.max("hn_pts_2y").alias("hn_pts_2y"),
                                            F.max("origami4_pts_2y").alias("origami4_pts_2y"),
                                            F.concat_ws(",", F.sort_array(F.collect_set("hn_src"))).alias("hn_sources"))
                .withColumn("hn_present", F.lit(True)).cache())

seg_order = ["H&N only", "H&N + Lung", "Lung only"]
acct_seg = (hn_set.join(lung_acct, "account_id", "full").join(account_level, "account_id", "left")
            .fillna(False, subset=["hn_present", "lung_present"]).fillna(0)
            .withColumn("account_level", F.coalesce("account_level", F.lit("UNKNOWN")))
            .withColumn("segment", F.when(F.col("hn_present") & F.col("lung_present"), "H&N + Lung")
                                    .when(F.col("hn_present"), "H&N only").otherwise("Lung only"))
            .cache())

tbl = acct_seg.groupBy("account_level").pivot("segment", seg_order).count().fillna(0)
tbl = tbl.withColumn("Total", sum(F.col(f"`{x}`") for x in seg_order))
tbl.unionByName(tbl.agg(*[F.sum(F.col(f"`{x}`")).alias(x) for x in seg_order + ["Total"]]).withColumn("account_level", F.lit("ALL"))) \
   .orderBy("account_level").show(truncate=False)
(acct_seg.groupBy("segment")
         .agg(F.count("*").alias("accounts"), F.sum("hn_hcps").alias("hn_hcps"),
              *[F.round(F.sum(x), 1).alias(x) for x in ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER"]])
         .withColumn("pct_accounts", F.round(100 * F.col("accounts") / F.sum("accounts").over(Window.partitionBy()), 1))
         .orderBy("segment").show(truncate=False))

## 7. Weekly DDD segmentation

### What this cell does: put every week's FASPRO DDD units into a segment
- **DDD rows:** FASPRO, summed per **account × week**, for weeks **starting** on or after 1 Jan 2026. The first week is the week of 5 Jan 2026.
- **Lung present that week** if either holds:
  - **historical signal:** the account is in the FASPRO Lung universe from the **non-DDD** sources (Xponent, MX, RX, McKesson, Chargemaster, YTD);
  - **13-week DDD rule:** the account bought FASPRO in **DDD** in the `LOOKBACK_WEEKS` (13) weeks **before** this week. The week's own purchase doesn't count.
  - DDD counts only through the 13-week rule. Otherwise every purchase would make its own account "Lung".
- **H&N present:** the H&N universe from section 6 (LE.AI + Chargemaster HNSCC + McKesson H&N diagnosis). It is the same every week.
- **Segment of the week:**
  - H&N + Lung / H&N only / Lung only;
  - **Unclassified**: neither, e.g. a first-ever buyer with no H&N HCPs;
  - **Unmapped account**: the DDD ID couldn't be placed on an L2/L3 account (open question Q9).
- **Outputs:**
  1. total units per segment;
  2. units per week per segment;
  3. the latest `RECENT_WEEKS` weeks per account;
  4. segment counts by L2 / L3 for the latest week.

In [ ]:
ddd_aw = (ddd.filter(F.col("week").between(F.lit(START_DATE), F.lit(END_DATE)))   # weeks starting in Jan 2026 or later
             .groupBy("account_id", "week").agg(F.round(F.sum("amount"), 3).alias("faspro_units")))

# 13-week DDD look-back (DDD only, weeks W-13 .. W-1)
ddd_hist = (ddd.filter("account_id IS NOT NULL").select(F.col("account_id").alias("a_acc"), F.col("week").alias("a_week")).distinct())
look = (ddd_aw.filter("account_id IS NOT NULL")
        .join(ddd_hist, (F.col("account_id") == F.col("a_acc")) &
                        (F.col("a_week") >= F.date_sub("week", LOOKBACK_WEEKS * 7)) & (F.col("a_week") <= F.date_sub("week", 7)), "left")
        .groupBy("account_id", "week").agg(F.count("a_week").alias("ddd_weeks_prior_13w")))

# historical Lung signal from the non-DDD sources (YTD)
hist_lung = (activity.filter((F.col("source") != "DDD") & F.col("account_id").isNotNull() & in_win)
                     .groupBy("account_id").agg(F.concat_ws(",", F.sort_array(F.collect_set("source"))).alias("hist_lung_sources")))

weekly = (ddd_aw.join(look, ["account_id", "week"], "left").join(hist_lung, "account_id", "left")
                .join(hn_set.select("account_id", "hn_present", "hn_hcps", "hn_sources"), "account_id", "left")
                .join(account_level, "account_id", "left")
                .fillna(False, subset=["hn_present"]).fillna(0, subset=["hn_hcps", "ddd_weeks_prior_13w"])
                .withColumn("lung_present", F.col("hist_lung_sources").isNotNull() | (F.col("ddd_weeks_prior_13w") > 0))
                .withColumn("segment", F.when(F.col("account_id").isNull(), "Unmapped account")
                                        .when(F.col("hn_present") & F.col("lung_present"), "H&N + Lung")
                                        .when(F.col("hn_present"), "H&N only")
                                        .when(F.col("lung_present"), "Lung only")
                                        .otherwise("Unclassified"))
                .cache())

all_segs = seg_order + ["Unclassified", "Unmapped account"]
print("1) FASPRO DDD units per segment (whole window)")
(weekly.groupBy("segment").agg(F.round(F.sum("faspro_units"), 1).alias("faspro_units"),
                               F.countDistinct("account_id").alias("accounts"), F.count("*").alias("account_weeks"))
       .withColumn("pct_units", F.round(100 * F.col("faspro_units") / F.sum("faspro_units").over(Window.partitionBy()), 1))
       .orderBy("segment").show(truncate=False))

print("2) FASPRO DDD units per week per segment")
weekly.groupBy("week").pivot("segment", all_segs).agg(F.round(F.sum("faspro_units"), 1)).fillna(0).orderBy("week").show(60, False)

last_weeks = [r["week"] for r in weekly.select("week").distinct().orderBy(F.desc("week")).limit(RECENT_WEEKS).collect()]
print(f"3) Latest {RECENT_WEEKS} weeks, per account:", [str(w) for w in sorted(last_weeks)])
(weekly.filter(F.col("week").isin(last_weeks))
       .select("week", "account_id", "account_level", "faspro_units", "segment", "hn_sources", "hist_lung_sources", "ddd_weeks_prior_13w")
       .orderBy(F.desc("week"), F.desc("faspro_units")).show(50, False))

latest = max(last_weeks) if last_weeks else None
print("4) Latest week", latest, "- accounts per segment by level")
(weekly.filter(F.col("week") == F.lit(latest)).groupBy(F.coalesce("account_level", F.lit("UNMAPPED")).alias("account_level"))
       .pivot("segment", all_segs).count().fillna(0).orderBy("account_level").show(truncate=False))

## 8. Account segment insights (L2/L3)

### What this cell does: build one insight table per account
- Starts from the historical account segments (section 6). Adds:
  - **account name**, from LE.AI's mapped account names where available;
  - **number of Lung sources** that show FASPRO use (DDD, Xponent, MX, RX, McKesson, Chargemaster): signal strength;
  - **weak Lung flag**: the only Lung signal is Chargemaster, which has no product column, so it may not be FASPRO.
- Prints a **scorecard per level × segment**:
  - accounts and % of accounts;
  - FASPRO DDD units YTD and % of units;
  - units per account;
  - H&N HCPs;
  - H&N treating patients and Origami-4 eligible patients (LE.AI, 2 years);
  - average number of Lung sources.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

SEG_COLORS = {"H&N only": "#4C78A8", "H&N + Lung": "#B279A2", "Lung only": "#54A24B",
              "Unclassified": "#BAB0AC", "Unmapped account": "#E45756"}
LUNG_SRC = ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER"]

acct_names = (coh.filter("cohort_account_id IS NOT NULL")
                 .select(F.col("cohort_account_id").alias("account_id"), F.col("cohort_account_name").alias("account_name"))
                 .dropDuplicates(["account_id"]))
ins = (acct_seg.join(acct_names, "account_id", "left")
       .withColumn("n_lung_sources", sum((F.col(x) > 0).cast("int") for x in LUNG_SRC))
       .withColumn("weak_lung_chargemaster_only", (F.col("n_lung_sources") == 1) & (F.col("CHARGEMASTER") > 0))
       .cache())

score = (ins.groupBy("account_level", "segment")
            .agg(F.count("*").alias("accounts"), F.round(F.sum("DDD"), 1).alias("faspro_ddd_units"),
                 F.sum("hn_hcps").alias("hn_hcps"), F.round(F.sum("hn_pts_2y"), 0).alias("hn_treating_pts_2y"),
                 F.round(F.sum("origami4_pts_2y"), 0).alias("origami4_eligible_2y"),
                 F.round(F.avg("n_lung_sources"), 2).alias("avg_lung_sources"))
            .toPandas())
score["pct_accounts"] = (100 * score.accounts / score.accounts.sum()).round(1)
score["pct_ddd_units"] = (100 * score.faspro_ddd_units / max(score.faspro_ddd_units.sum(), 1e-9)).round(1)
score["units_per_account"] = (score.faspro_ddd_units / score.accounts).round(2)
print(score.sort_values(["account_level", "segment"]).to_string(index=False))

### What this cell does: figures for the account segmentation
1. **Accounts per segment** (L2 vs L3) and **FASPRO DDD units per segment** (L2 vs L3).
2. **Weekly FASPRO DDD units per segment**, from section 7. It shows how the sales split evolves over time.
3. **Concentration (Pareto):** cumulative share of FASPRO DDD units vs number of accounts.
   - Also prints the share of units bought by the top 10 / 50 / 100 accounts.
4. **H&N + Lung accounts:** FASPRO units vs H&N treating patients.
   - Accounts top-right buy a lot **and** have many H&N patients: there, the Lung vs H&N split matters most after launch.

In [ ]:
segs = ["H&N only", "H&N + Lung", "Lung only"]
lv = score.pivot_table(index="segment", columns="account_level", values=["accounts", "faspro_ddd_units"], aggfunc="sum").fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, title in [(axes[0], "accounts", "Accounts per segment"), (axes[1], "faspro_ddd_units", "FASPRO DDD units YTD per segment")]:
    t = lv[metric].reindex(segs).fillna(0)
    t.plot(kind="bar", stacked=True, ax=ax, color=["#1f77b4", "#ff7f0e", "#999999"][:t.shape[1]])
    ax.set_title(title); ax.set_xlabel(""); plt.setp(ax.get_xticklabels(), rotation=0)
    for i, v in enumerate(t.sum(axis=1)):
        ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

wk = (weekly.groupBy("week").pivot("segment", list(SEG_COLORS)).agg(F.sum("faspro_units")).fillna(0)
            .orderBy("week").toPandas().set_index("week"))
fig, ax = plt.subplots(figsize=(13, 4))
for col in wk.columns:
    ax.plot(wk.index, wk[col], label=col, color=SEG_COLORS.get(col), linewidth=2)
ax.set_title("Weekly FASPRO DDD units by segment"); ax.set_ylabel("equivalized units"); ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

p = ins.filter("DDD > 0").select("account_id", "segment", "DDD").toPandas().sort_values("DDD", ascending=False)
p["cum_pct"] = 100 * p.DDD.cumsum() / p.DDD.sum()
for n in (10, 50, 100):
    if len(p) >= n:
        print(f"Top {n} accounts buy {p.cum_pct.iloc[n - 1]:.1f}% of mapped FASPRO DDD units (of {len(p):,} buying accounts)")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(p) + 1), p.cum_pct.values); ax.axhline(80, ls="--", color="grey")
ax.set_xlabel("accounts (ranked by FASPRO units)"); ax.set_ylabel("cumulative % of units"); ax.set_title("Concentration of FASPRO DDD")
plt.tight_layout(); plt.show()

sc = ins.filter("segment = 'H&N + Lung'").select("DDD", "hn_pts_2y", "account_level").toPandas()
fig, ax = plt.subplots(figsize=(6, 4))
for lvl, c in [("L2", "#1f77b4"), ("L3", "#ff7f0e")]:
    d = sc[sc.account_level == lvl]
    ax.scatter(d.DDD, d.hn_pts_2y, s=14, alpha=0.6, label=lvl, color=c)
ax.set_xlabel("FASPRO DDD units YTD"); ax.set_ylabel("H&N treating patients (LE.AI, 2y)")
ax.set_title("H&N + Lung accounts"); ax.legend()
plt.tight_layout(); plt.show()

### What this cell does: actionable account lists
1. **H&N only: launch targets.** Accounts with H&N HCPs and H&N patients but **no FASPRO use yet**, ranked by Origami-4 eligible patients, then H&N patients.
2. **H&N + Lung: where the split matters most.** Ranked by FASPRO DDD units, with their H&N patients.
   - These are the accounts whose sales will need the Lung vs H&N split (Careset) after launch.
3. **Lung only: top FASPRO buyers with no H&N signal.** Their sales stay Lung.
4. **Signal strength:** accounts per segment by number of Lung sources.
   - Accounts whose only Lung signal is Chargemaster are flagged as weak.
- Each list shows the top 20. The full table is `ins`, saved as `HN_FASPRO_ACCOUNT_INSIGHTS`.

In [ ]:
cols = ["account_id", "account_name", "account_level", "hn_hcps", "hn_pts_2y", "origami4_pts_2y", "DDD", "XPONENT", "MX",
        "n_lung_sources", "hn_sources"]
print("1) H&N only - launch targets (no FASPRO use yet)")
ins.filter("segment = 'H&N only'").orderBy(F.desc("origami4_pts_2y"), F.desc("hn_pts_2y")).select(*cols).show(20, False)
print("2) H&N + Lung - largest FASPRO buyers with H&N exposure")
ins.filter("segment = 'H&N + Lung'").orderBy(F.desc("DDD")).select(*cols).show(20, False)
print("3) Lung only - largest FASPRO buyers without an H&N signal")
ins.filter("segment = 'Lung only'").orderBy(F.desc("DDD")).select(*cols).show(20, False)
print("4) Signal strength - accounts by number of Lung sources")
(ins.filter("segment != 'H&N only'").groupBy("segment").pivot("n_lung_sources").count().fillna(0).orderBy("segment").show())
print("Accounts whose only Lung signal is Chargemaster (no product column):",
      ins.filter("weak_lung_chargemaster_only").count())

## 9. Save

### What this cell does: optionally save the results to Dataiku
- Leave `SAVE = False` to only look at the results.
- To save: create empty output datasets in the Flow with these names, then set `SAVE = True`.
  - `HN_FASPRO_ACCOUNT_SEGMENTS`: one row per L2/L3 account with its historical (YTD) segment and FASPRO activity per source.
  - `HN_FASPRO_DDD_WEEKLY_SEGMENTS`: one row per account × week with FASPRO DDD units and that week's segment (the main output).
  - `HN_FASPRO_ACCOUNT_INSIGHTS`: one row per account with segment, name, H&N volumes, Lung sources and the weak-signal flag.

In [ ]:
SAVE = False
outputs = {"HN_FASPRO_ACCOUNT_SEGMENTS": acct_seg, "HN_FASPRO_DDD_WEEKLY_SEGMENTS": weekly,
           "HN_FASPRO_ACCOUNT_INSIGHTS": ins}
if SAVE:
    for name, d in outputs.items():
        dkuspark.write_with_schema(dataiku.Dataset(name), d)
        print("saved", name)